In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [3]:
df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df.shape

(5572, 5)

In [5]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')

In [6]:
df=df[['v1','v2']]
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.columns=['label','message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [9]:
df=df.dropna()

In [10]:
df.duplicated().sum()

np.int64(403)

In [11]:
df=df.drop_duplicates()

In [12]:
df.shape

(5169, 2)

In [13]:
df['label']=df['label'].map({'ham':0,'spam':1})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
df['label'].value_counts()

label
0    4516
1     653
Name: count, dtype: int64

In [15]:
X=df['message']
y=df['label']

In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [17]:
X_train.shape

(4135,)

In [18]:
X_test.shape

(1034,)

In [19]:
y_train.shape

(4135,)

In [20]:
y_test.shape

(1034,)

In [21]:
vectorizer=TfidfVectorizer(lowercase=True,stop_words='english')

In [22]:
X_train_tfidf=vectorizer.fit_transform(X_train)
print('Train shape:',X_train_tfidf.shape)

Train shape: (4135, 7329)


In [23]:
X_test_tfidf=vectorizer.transform(X_test)
print('Testing shape:',X_test_tfidf.shape)

Testing shape: (1034, 7329)


In [24]:
model=MultinomialNB()
model.fit(X_train_tfidf,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [25]:
y_pred=model.predict(X_test_tfidf)


In [26]:
y_pred

array([0, 0, 0, ..., 1, 0, 0], shape=(1034,))

In [27]:
accuracy=accuracy_score(y_test,y_pred)

In [28]:
accuracy

0.9661508704061895

In [29]:
cr=classification_report(y_test,y_pred,target_names=['Not spam','spam'])

In [30]:
print(cr)

              precision    recall  f1-score   support

    Not spam       0.96      1.00      0.98       903
        spam       0.99      0.74      0.85       131

    accuracy                           0.97      1034
   macro avg       0.98      0.87      0.91      1034
weighted avg       0.97      0.97      0.96      1034



In [31]:
cm=confusion_matrix(y_test,y_pred)

In [32]:
print(cm)

[[902   1]
 [ 34  97]]


In [34]:
with open('../model/spam_model.pkl','wb') as file:
    pickle.dump(model,file)
print('model saved successfully!')    

model saved successfully!


In [36]:
with open('../model/vectorizer.pkl','wb') as file:
    pickle.dump(vectorizer,file)
print('Vectorizer saved successfully')    

Vectorizer saved successfully


In [37]:
with open("../model/spam_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

with open("../model/vectorizer.pkl", "rb") as file:
    loaded_vectorizer = pickle.load(file)

print("Model and vectorizer loaded successfully!")

Model and vectorizer loaded successfully!


In [38]:
new_message = [
    "Congratulations! You have won a free prize!"
]

new_message_vector = loaded_vectorizer.transform(new_message)

prediction = loaded_model.predict(new_message_vector)[0]

if prediction == 1:
    print("Prediction: Spam")
else:
    print("Prediction: Not Spam")

Prediction: Spam


In [39]:
new_message = [
    "Hey, are you coming to college today?"
]

new_message_vector = loaded_vectorizer.transform(new_message)

prediction = loaded_model.predict(new_message_vector)[0]

if prediction == 1:
    print("Prediction: Spam")
else:
    print("Prediction: Not Spam")

Prediction: Not Spam
